### Rudimentary (Deprecated way to look for Accuracy for each OD)

In [22]:
import numpy as np
import matplotlib.pyplot as plt

def hit_judgment(timing_error, OD=8):
    w_300 = 80-6*OD
    w_100 = 140-8*OD
    w_50 = 200-10*OD
    if abs(timing_error) <= w_300:
        return 300
    elif abs(timing_error) <= w_100:
        return 100
    elif abs(timing_error) <= w_50:
        return 50
    else:
        return 0

# Define parameters of the normal distribution
mu = 5  # Mean
UR = 150
sigma = UR*.1  # Standard deviation
OD = 9
size = 1000
# Generate 1000 points following the normal distribution
points = np.random.normal(loc=mu, scale=sigma, size=size)

acc = 100*sum([hit_judgment(x, OD=OD) for x in points])/(size*300)
print(acc)

93.26666666666667


### 1) Find the UR from Accuracy 2) Find the Accuracy from UR

In [243]:
from scipy.stats import norm

In [244]:
def calculate_UR(OD, accuracy):
    w_300 = 80 - 6 * OD  # 300 hit window
    z = norm.ppf((1 + accuracy) / 2)  # Inverse CDF for the desired probability
    sigma = w_300 / z  # Standard deviation
    UR = sigma*10  # Variance
    return UR

def calculate_accuracy(OD, UR):
    w_300 = 80 - 6 * OD
    sigma=UR/10
    k = norm.cdf(-w_300,loc=0, scale=sigma)
    accuracy = 1-2*k
    return accuracy

def convert_accuracy(OD_prior, OD_later, accuracy):
    return calculate_accuracy(OD_later,calculate_UR(OD_prior,accuracy))

In [245]:
# Example calculations
od_values = [8, 9, 10]
accuracy = [0.96, 0.99]

for od in od_values:
    for p in accuracy:
        var = calculate_UR(od, p)
        print(f"OD: {od}, P: {p*100}%, Unstable Rate: {var:.2f}")

OD: 8, P: 96.0%, Unstable Rate: 155.81
OD: 8, P: 99.0%, Unstable Rate: 124.23
OD: 9, P: 96.0%, Unstable Rate: 126.60
OD: 9, P: 99.0%, Unstable Rate: 100.94
OD: 10, P: 96.0%, Unstable Rate: 97.38
OD: 10, P: 99.0%, Unstable Rate: 77.64


### Elo Updating Formulas

In [249]:
K, D, W = 500, 500, 10
# K: every 500 difference, the winrate increases tenfold

In [247]:
def expected_score(elo1, elo2):
    return 1/(1+W**((elo2-elo1)/D)), 1/(1+W**((elo1-elo2)/D))

def update_elo(elo1, elo2, s1, s2):
    e1, e2 = expected_score(elo1,elo2)
    return elo1 + K*(s1-e1), elo2 + K*(s2-e2)

In [251]:
# Examples
expected_score(3000, 3000)
update_elo(9000, 6000, 1, 0), update_elo(9000, 6000, 0, 1)

((9000.0004999995, 5999.9995000005), (8500.0004999995, 6499.9995000005))

### Getting the win value from the accuracy (player x chart ELO updating)

In [256]:
import numpy as np

In [253]:
getting_win_func=lambda accuracy: accuracy**10

In [254]:
x_values = np.concatenate([np.arange(0, 0.9, 0.05), np.arange(0.9, 1.0051, 0.005)])
y_values = getting_win_func(x_values)  # Corresponding y values
lookup_table = [[x,y] for x, y in zip(x_values, y_values)]

In [279]:
def get_win_from_acc(accuracy):
    index=0
    while lookup_table[index][0] <= accuracy: index+=1
    alpha = (accuracy-lookup_table[index-1][0])/(lookup_table[index][0]-lookup_table[index-1][0])
    return min((1-alpha)*lookup_table[index-1][1]+(alpha)*lookup_table[index][1],1)

In [289]:
a=0.956
get_win_from_acc(a), a**10

(0.6377715910879758, 0.6376449474101403)

### Chart and Player Class Definitions

In [290]:
class Chart:
    def __init__(self, stars=1, chart_type=None, animosity=0, OD=9):
        """Represents a specific osu! chart/map."""
        self.stars = stars  # Number of stars the map has
        self.chart_type = chart_type if chart_type else []  # List of characteristics, e.g., ["speed", "reading"]
        self.animosity = animosity  # Unexpected difficulty modifier, range [0, 300], 0 if farm/not tournment, 1000 if you are expected to 0% accuracy
        self.OD = OD # OD from 0 to 11
        self.expected_OD = max(-((self.stars-9)**2)/9 + 10,min(self.stars+1,10))
        # self.expected_UR = calculate_UR(self.expected_OD, 0.97 * (1 - self.animosity/1000))
        self.chart_elo = 1000 * stars * (1+self.animosity/1000)

    def __str__(self):
        return (
            f"Stars: {self.stars}\n"
            f"Chart Type: {self.chart_type if self.chart_type else 'N/A'}\n"
            f"Animosity (>50 -> Tournment): {self.animosity}\n"
            f"OD, EOD: {self.OD}, {self.expected_OD}\n"
            f"Chart ELO: {self.chart_elo}\n"
            )

In [291]:
class OsuPlayer:
    def __init__(self, name, elo_rating=0,rank_rating=9999999):
        """Represents an osu! player with stats and performance metrics."""
        self.name = name
        self.star_rating = 1  # Average star rating of farm maps (animosity 0 maps)
        self.tournament_star_rating = 1  # Average star rating played in tournaments
        self.rank_rating = rank_rating  # Standard osu! global rank (lower is better)

        self.elo_rating = elo_rating

        self.effective_rank = None  # Effective rank estimation

        self.total_matches = 0
        self.win_matches = 0

    def update_player_elo(self, target, performance):
        """Calculate ELO"""
        match target.__class__.__name__:
            case "Chart":
                # get_win_from_acc = lambda accuracy: accuracy**10
                converted_performance=get_win_from_acc(convert_accuracy(target.OD, target.expected_OD, performance))
                self.elo_rating, _ = update_elo(self.elo_rating, target.chart_elo, converted_performance, 1-converted_performance)
                return [self.elo_rating, target.chart_elo]
            case "OsuPlayer":
                self.elo_rating, target.elo_rating = update_elo(self.elo_rating, target.elo_rating, performance, 1-performance)
                return [self.elo_rating, target.elo_rating]
        pass  # Placeholder

    def update_rank_estimate(self):
        """Update the player's effective rank based on skill metrics."""
        pass  # Placeholder

    def update_star_ratings(self, chart, accuracy):
        """Update star ratings based on solo play metrics."""
        return "a"
        # If tournament chart, update tournament stars similarly
        if chart.animosity >= 50:
            return "a"
        
    def __str__(self):
        return (
            f"Player: {self.name}\n"
            f"Standard Star Rating: {self.star_rating}\n"
            f"Tournament Star Rating: {self.tournament_star_rating}\n"
            f"Rank Rating: {self.rank_rating}\n"
            f"Effective Rank: {self.effective_rank if self.effective_rank else 'N/A'}\n"
            f"ELO Rating: {self.elo_rating}\n"
        )

### Testing

In [294]:
aerora=OsuPlayer(name="Aerora", elo_rating=1000, rank_rating=450)
mugenq=OsuPlayer(name="Mugenq", elo_rating=1000, rank_rating=130000)
tech=Chart(stars=7,OD=9,animosity=10,chart_type=["tapping"])

aerora.update_player_elo(target=mugenq, performance=1)
aerora.update_player_elo(target=tech, performance=0.98)

[1573.7898249965228, 7070.0]

In [318]:

# Create a player with initial stars at 5
player = OsuPlayer(name="TestPlayer99", elo_rating=5000)

# Simulate playing 10 charts with OD 9 and stars 7
charts = [Chart(stars=4,OD=7,animosity=0,chart_type=["tapping"]) for _ in range(10)]

print(charts[0])

# Update player's star rating based on charts played
for chart in charts:
    player.update_player_elo(chart, performance=0.97)

print(player)  # Output the player's stats after updates


Stars: 4
Chart Type: ['tapping']
Animosity (>50 -> Tournment): 0
OD, EOD: 7, 7.222222222222222
Chart ELO: 4000.0

Player: TestPlayer99
Standard Star Rating: 1
Tournament Star Rating: 1
Rank Rating: 9999999
Effective Rank: N/A
ELO Rating: 4183.5071807904405



### To do
1) fix 5000 elo dropping when not SSing lower star maps
2) incorporate mods

In [27]:
def modulo_power(base, exponent, modulo):
    result = 1
    base = base % modulo  # Handle base greater than modulo
    while exponent > 0:
        if exponent % 2 == 1:  # If the exponent is odd
            result = (result * base) % modulo
        base = (base * base) % modulo  # Square the base
        exponent //= 2  # Divide the exponent by 2
    return result

# Test the function
m=25
e=2**3+1
modulos = [7, 11, 13, 101, 103, 87, 97, 37, 31]

print("e=",e)
print("modulos=",modulos)
print("cyphertexts=",[modulo_power(m, e, N) for N in modulos])


e= 9
modulos= [7, 11, 13, 101, 103, 87, 97, 37, 31]
cyphertexts= [1, 4, 12, 68, 79, 16, 27, 36, 1]
